# VetFlow — Protótipo do Componente de IA (Sprint 3)

**Objetivo:** demonstrar, de forma simplificada e com dados sintéticos, a abordagem de IA escolhida para o VetFlow — um **modelo preditivo** que estima o risco de abandono de cuidado (vacina / retorno / tratamento) de cada pet e gera uma lista priorizada de ação para a clínica.

Não vamos utilizar **dados reais de produção**. Vamos simular o formato dos dados que já existem no banco Oracle do VetFlow (perfil do pet, histórico de vacinas, consultas, medicações e comportamento) para provar a viabilidade técnica da abordagem antes da integração real com as APIs Java (Spring Boot) e .NET.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, classification_report

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)
N_PETS = 400

## 1) Simulação dos dados

Equivalente ao que já existe no Oracle do VetFlow: perfil do pet, histórico de vacinas, consultas, medicações e comportamento.

In [ ]:
species = rng.choice(["cachorro", "gato"], size=N_PETS, p=[0.7, 0.3])
breed_size = rng.choice(["pequeno", "medio", "grande"], size=N_PETS, p=[0.4, 0.35, 0.25])
age_months = rng.integers(2, 168, size=N_PETS)

# intervalo esperado entre vacinas varia por espécie/porte (regra de negócio real)
base_interval = np.select(
    [species == "gato", breed_size == "grande"],
    [365, 300],
    default=365,
)
expected_vaccine_interval_days = base_interval + rng.integers(-15, 15, size=N_PETS)

days_since_last_vaccine = rng.integers(10, 600, size=N_PETS)
days_since_last_consultation = rng.integers(10, 500, size=N_PETS)
missed_appointments_count = rng.poisson(0.6, size=N_PETS)
active_medication = rng.choice([0, 1], size=N_PETS, p=[0.75, 0.25])
adherence_score = np.clip(rng.normal(0.7, 0.2, size=N_PETS), 0, 1)  # adesão a tratamentos
behavior_alerts = rng.poisson(0.4, size=N_PETS)  # registros de comportamento preocupante

df = pd.DataFrame({
    "species": species,
    "breed_size": breed_size,
    "age_months": age_months,
    "expected_vaccine_interval_days": expected_vaccine_interval_days,
    "days_since_last_vaccine": days_since_last_vaccine,
    "days_since_last_consultation": days_since_last_consultation,
    "missed_appointments_count": missed_appointments_count,
    "active_medication": active_medication,
    "adherence_score": adherence_score.round(2),
    "behavior_alerts": behavior_alerts,
})

df["vaccine_overdue_ratio"] = (
    df["days_since_last_vaccine"] / df["expected_vaccine_interval_days"]
).round(2)

df.head()

## 2) Rótulo histórico simulado

Em produção, viria do histórico real: pets que de fato deixaram vacina vencer, abandonaram tratamento etc.

In [ ]:
risk_score_latente = (
    2.2 * (df["vaccine_overdue_ratio"] > 1.0).astype(int)
    + 0.9 * df["missed_appointments_count"]
    + 1.3 * (df["adherence_score"] < 0.5).astype(int)
    + 0.5 * df["behavior_alerts"]
    + 0.6 * (df["days_since_last_consultation"] > 240).astype(int)
    + rng.normal(0, 0.8, size=N_PETS)
)
df["abandono_futuro"] = (risk_score_latente > risk_score_latente.mean()).astype(int)
df["abandono_futuro"].value_counts()

## 3) Treino do modelo preditivo

Classificação binária, interpretável (regressão logística).

In [ ]:
feature_cols = [
    "age_months",
    "vaccine_overdue_ratio",
    "days_since_last_consultation",
    "missed_appointments_count",
    "active_medication",
    "adherence_score",
    "behavior_alerts",
]
X = df[feature_cols]
y = df["abandono_futuro"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_SEED, stratify=y
)

modelo = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(random_state=RANDOM_SEED)),
])
modelo.fit(X_train, y_train)

y_pred = modelo.predict(X_test)
y_proba = modelo.predict_proba(X_test)[:, 1]

print("=" * 70)
print("VALIDAÇÃO DO MODELO (dados sintéticos, apenas para prova de conceito)")
print("=" * 70)
print(f"ROC-AUC: {roc_auc_score(y_test, y_proba):.3f}")
print(classification_report(y_test, y_pred, target_names=["baixo risco", "alto risco"]))

In [ ]:
coefs = pd.Series(modelo.named_steps["clf"].coef_[0], index=feature_cols).sort_values()
print("Peso de cada fator no risco (coeficientes do modelo):")
coefs.round(2)

## 4) Personalização — mesmo "atraso em dias", risco diferente por pet

O modelo não aplica uma regra fixa igual para todos os pets.

In [ ]:
exemplo = pd.DataFrame([
    {  # filhote pequeno com atraso curto, mas proporcionalmente grande
        "age_months": 6, "vaccine_overdue_ratio": 1.15,
        "days_since_last_consultation": 200, "missed_appointments_count": 1,
        "active_medication": 0, "adherence_score": 0.8, "behavior_alerts": 0,
    },
    {  # idoso de grande porte, mesmo atraso relativo, contexto mais grave
        "age_months": 130, "vaccine_overdue_ratio": 1.15,
        "days_since_last_consultation": 200, "missed_appointments_count": 3,
        "active_medication": 1, "adherence_score": 0.4, "behavior_alerts": 1,
    },
])
exemplo["score_risco"] = modelo.predict_proba(exemplo[feature_cols])[:, 1].round(3)
print(exemplo[["age_months", "vaccine_overdue_ratio", "adherence_score", "score_risco"]].to_string(index=False))
print(
    "\n-> Mesmo com o MESMO atraso proporcional de vacina (15% acima do esperado),"
    "\n   o pet idoso com medicação ativa e menor adesão recebe um score de risco"
    "\n   mais alto — é a IA personalizando a prioridade por pet, não aplicando"
    "\n   uma regra fixa igual para todos."
)

## 5) Saída: lista priorizada para a clínica

O que o serviço de IA devolveria para a API consumir e exibir no app / painel da clínica.

In [ ]:
df["score_risco"] = modelo.predict_proba(X[feature_cols])[:, 1].round(3)
prioridade = df.sort_values("score_risco", ascending=False).head(10).reset_index(drop=True)
prioridade.index += 1
prioridade.to_csv("priority_list_sample.csv", index_label="prioridade")

print("LISTA PRIORIZADA PARA A CLÍNICA (amostra sintética)")
prioridade[["species", "breed_size", "age_months", "vaccine_overdue_ratio", "score_risco"]]